# Modelo ML XGBOOST CLASSIFIER

In [101]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import json


train_data = pd.read_csv("../models/x_train_sel.csv")
test_data = pd.read_csv("../models/x_test_sel.csv")

X_train = train_data.drop(["ciudad"], axis=1)
y_train = train_data["ciudad"]
X_test = test_data.drop(["ciudad"], axis=1)
y_test = test_data["ciudad"]

train_data.head()


,airline_n,category,clasificacion_destino_n,class_n,desc_actual_n,distance_to_city_center_km,entornos_n,estimated_price_eur_x,estimated_price_eur_y,flight_duration_hr,...,humedad_actual,perfil_viajero_n,popularidad_reddit,precipitacion,stops_n,temp_actual,temp_max,temp_min,temporada_n,ciudad
0,8.0,4.0,2.0,1.0,0.0,0.23,0.0,694.11,227.27,2.2,...,67.0,1.0,100.0,0.0,2.0,13.3,18.4,10.1,1.0,2
1,5.0,4.0,11.0,1.0,0.0,0.19,0.0,1317.75,290.86,8.7,...,38.0,6.0,100.0,0.0,2.0,21.2,14.9,4.1,2.0,49
2,1.0,3.0,1.0,3.0,2.0,0.19,0.0,2413.91,124.68,2.6,...,36.0,0.0,100.0,0.0,0.0,24.2,16.1,5.7,1.0,76
3,5.0,5.0,2.0,0.0,4.0,0.12,0.0,201.77,444.47,3.0,...,71.0,6.0,100.0,0.0,1.0,28.8,29.6,18.0,0.0,52
4,9.0,4.0,2.0,1.0,8.0,0.23,0.0,811.27,207.86,12.8,...,94.0,0.0,100.0,0.0,2.0,15.2,18.4,10.1,2.0,46


In [102]:

# 4. Entrenar modelo
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train, y_train)

# 5. Evaluar
y_pred = model.predict(X_test)
print("✔️ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Clasificación:\n", classification_report(y_test, y_pred))

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [20:20:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


✔️ Accuracy: 1.0

📊 Clasificación:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       818
           1       1.00      1.00      1.00       789
           2       1.00      1.00      1.00       415
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00        38
           5       1.00      1.00      1.00       827
           6       1.00      1.00      1.00       491
           7       1.00      1.00      1.00       712
           8       1.00      1.00      1.00       682
           9       1.00      1.00      1.00        20
          10       1.00      1.00      1.00       617
          11       1.00      1.00      1.00       802
          12       1.00      1.00      1.00       513
          13       1.00      1.00      1.00       633
          14       1.00      1.00      1.00        38
          15       1.00      1.00      1.00        29
          16       1.00      1.00      1.00  

# 🧪 PREDICCIÓN PERSONALIZADA

In [103]:
X_train.columns.to_frame().to_csv("../data/processed/x_train_columns.csv", index=False, header=False)

# PASO 2: Definir función para construir entrada de usuario

import pandas as pd

def construir_input_usuario(valores_activados, columnas_referencia_path):
    columnas_modelo = pd.read_csv(columnas_referencia_path, header=None).squeeze().tolist()
    input_df = pd.DataFrame(columns=columnas_modelo)
    input_df.loc[0] = 0
    for col, val in valores_activados.items():
        if col in input_df.columns:
            input_df.at[0, col] = val
    return input_df

# PASO 3: Simular selección del usuario
seleccion_usuario = {
    "perfil_Familia": 1,
    "entorno_Ciudad": 1,
    "Clasificacion_Relax": 1,
    "temporada_Verano": 1,
    "origin_city_París": 1,
    "flight_price": 120,
    "estimated_price_eur_y": 90,
    "temp_max": 30,
    "temp_min": 20
}

X_user = construir_input_usuario(
    seleccion_usuario,
    columnas_referencia_path="../data/processed/x_train_columns.csv"
)

with open("../data/processed/Json/ciudad_transformation_rules.json", "r") as f:
    ciudad_mapping = json.load(f)

id_to_ciudad = {str(v): k for k, v in ciudad_mapping.items()}

y_train_original = pd.read_excel("../data/processed/X&Ys/y_train.xlsx").squeeze()
y_train_original = y_train_original.astype(str).map(id_to_ciudad)

le = LabelEncoder()
le.fit(y_train_original)


LabelEncoder()

In [105]:
probs = model.predict_proba(X_user)[0]
top5_indices = np.argsort(probs)[::-1][:5]
top5_labels = model.classes_[top5_indices]
top5_ciudades = le.inverse_transform(top5_labels)

print("🏝️ Top 5 ciudades recomendadas:", top5_ciudades)

🏝️ Top 5 ciudades recomendadas: ['perth' 'amman' 'vilnius' 'marrakech' 'calgary']


# 🌍 FUNCIONES DE ENRIQUECIMIENTO (usando df original)

In [121]:
full_df = pd.read_csv("../data/processed/total_data_240k.csv")

temporada_usuario = "Verano" 

def get_clima_estimado(ciudad, temporada):
    clima = full_df[
        (full_df['ciudad'] == ciudad) &
        (full_df['temporada'].str.strip().str.lower() == temporada.strip().lower())
    ]
    if clima.empty:
        return ""
    datos = clima[['temp_max', 'temp_min', 'precipitacion']].mean().round(1).to_dict()
    return f"{datos}"


def get_eventos(ciudad, temporada):
    eventos = full_df[
        (full_df['ciudad'] == ciudad) &
        (full_df['temporada'].str.strip().str.lower() == temporada.strip().lower())
    ][['evento_nombre', 'evento_categoria', 'evento_desc', 'fecha']]
    eventos = eventos.dropna().drop_duplicates().head(3)
    if eventos.empty:
        return ""
    return "\n".join(
        f"- {row['evento_nombre']} ({row['evento_categoria']}): {row['evento_desc']} [{row['fecha']}]"
        for _, row in eventos.iterrows()
    )

def get_precio_vuelo(origen, destino):
    vuelos = full_df[(full_df['origin_city'] == origen) & (full_df['ciudad'] == destino)]
    return round(vuelos['flight_price'].mean(), 2) if not vuelos.empty else "Sin datos"

def get_hotel(ciudad):
    hoteles = full_df[full_df['ciudad'] == ciudad][['hotel_name', 'estimated_price_eur_y', 'hotel_type', 'distance_to_city_center_km']]
    hotel = hoteles.dropna().sort_values(by='estimated_price_eur_y').head(1)
    return hotel.to_dict(orient='records')[0] if not hotel.empty else "Sin hoteles"


# 📦 MOSTRAR INFO ENRIQUECIDA

In [123]:
for ciudad in top5_ciudades:
    print(f"\n🌍 Ciudad: {ciudad}")
    clima = get_clima_estimado(ciudad, temporada_usuario)
    if clima:
        print("☁️ Clima estimado:", clima)

    eventos = get_eventos(ciudad, temporada_usuario)
    if eventos:
        print("🎫 Eventos:\n", eventos)

    print("✈️ Vuelo desde origen:", get_precio_vuelo("Madrid", ciudad))
    print("🏨 Hotel recomendado:", get_hotel(ciudad))


🌍 Ciudad: perth
☁️ Clima estimado: {'temp_max': 18.4, 'temp_min': 10.1, 'precipitacion': 0.0}
🎫 Eventos:
 - Money Event (Desconocido): Sin descripción [2026-01-01]
- Event (Desconocido): Sin descripción [2025-10-21]
- Event (Desconocido): Sin descripción [2025-10-20]
✈️ Vuelo desde origen: 1207.97
🏨 Hotel recomendado: {'hotel_name': 'Sunset Perth Haven', 'estimated_price_eur_y': 284.17, 'hotel_type': 'Eco-Lodge', 'distance_to_city_center_km': 0.1}

🌍 Ciudad: amman
✈️ Vuelo desde origen: 1137.23
🏨 Hotel recomendado: {'hotel_name': 'Paradise Amman Resort', 'estimated_price_eur_y': 277.12, 'hotel_type': 'Business Hotel', 'distance_to_city_center_km': 0.11}

🌍 Ciudad: vilnius
☁️ Clima estimado: {'temp_max': 2.8, 'temp_min': -0.3, 'precipitacion': 1.6}
🎫 Eventos:
 - sin_evento (no_aplica): no_aplica [2024-12-31]
- sin_evento (no_aplica): no_aplica [2024-12-30]
- sin_evento (no_aplica): no_aplica [2024-12-29]
✈️ Vuelo desde origen: 816.5
🏨 Hotel recomendado: {'hotel_name': 'Ocean Vilnius Su